In [ ]:
!pip install -q numerapi 'pandas>=2.3.3,<3' lightgbm pyarrow scikit-learn scipy matplotlib
# make sure you restart your kernel session before continuing

In [ ]:
from numerapi import SignalsAPI
import pandas as pd

DATASET_VERSION = "v3.0"
DATASET_DIR = f"signals/{DATASET_VERSION}"
TARGET = "target_jupiter_60"

napi = SignalsAPI()

In [ ]:
napi.download_dataset(f"{DATASET_DIR}/train.parquet")
napi.download_dataset(f"{DATASET_DIR}/validation.parquet")

In [ ]:
train = pd.read_parquet(f"{DATASET_DIR}/train.parquet")
validation = pd.read_parquet(f"{DATASET_DIR}/validation.parquet")

# Tickers

The Signals dataset contains two tickers: `numerai_ticker` and `composite_figi`:
- `numerai_ticker` is given for the entire history
- `composite_figi` only goes back to September 2022.

In [ ]:
import matplotlib.pyplot as plt

def plot_ticker_counts_per_date(df, title):
    dated = df.assign(date=pd.to_datetime(df['date']))

    # Count unique 'numerai_ticker' and 'composite_figi' per 'date'
    nticker_count_per_date = dated.groupby('date')['numerai_ticker'].nunique().reset_index(name='numerai_ticker_count')
    figi_count_per_date = dated.groupby('date')['composite_figi'].nunique().reset_index(name='figi_count')

    # Merge the counts into a single DataFrame for plotting
    merged_counts = pd.merge(nticker_count_per_date, figi_count_per_date, on='date')

    # Plotting
    plt.figure(figsize=(10, 6))
    plt.plot(merged_counts['date'], merged_counts['numerai_ticker_count'], label='Unique Numerai Tickers', marker='o')
    plt.plot(merged_counts['date'], merged_counts['figi_count'], label='Unique Composite FIGIs', marker='x')

    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel('Count')
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_ticker_counts_per_date(validation, 'Validation Dataset numerai_ticker and composite_figi Counts per Date')

If you have Bloomberg tickers, you can map to `numerai_ticker` by replacing the exchange code with the ISO country code

In [ ]:
import random

# Computed using https://stockmarketmba.com/globalstockexchanges.php
# and https://www.isin.net/country-codes/
# Converting Bloomberg exchange code -> Country -> ISO 3166
TICKER_CTRY_MAP = {
    "AU": "AU", "AV": "AT", "BB": "BE", "BZ": "BR", "CA": "CA",
    "CB": "CO", "CH": "CN", "CI": "CL", "CN": "CA", "CP": "CZ",
    "DC": "DK", "EY": "EG", "FH": "FI", "FP": "FR", "GA": "GR",
    "GR": "DE", "GY": "DE", "HB": "HU", "HK": "HK", "ID": "IE",
    "IJ": "ID", "IM": "IT", "IN": "IN", "IT": "IL", "JP": "JP",
    "KS": "KR", "LN": "GB", "MF": "MX", "MK": "MY", "NA": "NL",
    "NO": "NO", "NZ": "NZ", "PE": "PE", "PL": "PT", "PM": "PH",
    "PW": "PL", "QD": "QA", "RM": "RU", "SJ": "ZA", "SM": "ES",
    "SP": "SG", "SS": "SE", "SW": "CH", "TB": "TH", "TI": "TR",
    "TT": "TW", "UH": "AE", "US": "US", "UQ": "US",
}

def map_country_code(row):
    if row["bloomberg_ticker"] is None:
        return None
    split_ticker = row["bloomberg_ticker"].split()
    if len(split_ticker) < 2:
        print(f'No country code for {row["bloomberg_ticker"]}')
        return None

    ticker = split_ticker[0]
    country_code = split_ticker[-1]
    iso_country_code = TICKER_CTRY_MAP.get(country_code)
    return f"{ticker} {iso_country_code}"

# create test dataframe with Bloomberg tickers
df = pd.DataFrame([
  {'bloomberg_ticker': '000640 KS', 'signal': random.random()},
  {'bloomberg_ticker': '1103 TT', 'signal': random.random()},
  {'bloomberg_ticker': 'A2A IM', 'signal': random.random()},
  {'bloomberg_ticker': 'ABBN SW', 'signal': random.random()}
])

# convert to numerai_ticker
df['numerai_ticker'] = df.apply(
    map_country_code, axis=1
)

assert df.iloc[0]['numerai_ticker'] == '000640 KR'
assert df.iloc[1]['numerai_ticker'] == '1103 TW'
assert df.iloc[2]['numerai_ticker'] == 'A2A IT'
assert df.iloc[3]['numerai_ticker'] == 'ABBN CH'

# Features

Features with `{n}(d|w)` in the name (for example, `feature_adv_20d_factor`) are time-series features that are computed over `n` days or `n` weeks.

Features with `country_ranknorm` in the name are grouped by country, then ranked, then gaussianized.

Features with `factor` in the name refer to risk factors that most of the targets are neutral to.

PPO, RSI and TRIX are examples of technical indicators.

PPO is a percentage price oscillator that compares shorter and longer moving averages in a ratio
RSI is the relative strength index usually used as an overbought/oversold indicator
TRIX is a triple exponential moving average indicator usually used as momentum or reversal feature

`momentum_52w_less_4w` refers to one year return of a stock excluding the last 4 weeks.



In [ ]:
train.filter(like="feature_").columns.tolist()

# Modeling

The dataset includes a small set of starter features that you can combine with your own data. This example trains a model on the v3 Jupiter 60-day target. The starter features alone are not expected to produce a competitive Signals model.

In [ ]:
import lightgbm as lgb

feature_cols = [
    col for col in train.columns
    if col.startswith("feature_") and col != "feature_country"
]

# https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=5,
    num_leaves=2**5-1,
    colsample_bytree=0.1,
)

# This will take a few minutes 🍵
model.fit(
    train[feature_cols],
    train[TARGET],
)

# Scores

The v3 scores **Neutral Correlation (NCORR)** and **Neutral Meta Model Contribution (NMMC)** use the Jupiter 60-day target and the v3 neutralizer. We calculate them with `numerai-tools` below. The public data does not include the v3 neutral stake-weighted Meta Model, so the contribution calculation uses the published validation example predictions as a **reference proxy**. It demonstrates the NMMC method, but its values are **not official NMMC scores**.

Download the v3 neutralizer and example predictions. Unlike v2.1, v3 has no sample-weight file.

In [ ]:
napi.download_dataset(f"{DATASET_DIR}/validation_neutralizer.parquet")
napi.download_dataset(f"{DATASET_DIR}/validation_example_preds.parquet")

Here, we embargo the first 12 eras, then run the next 50 eras. The full validation period will take a long time to complete, so we suggest using the Signals Diagnostics tool to calculate the full validation metrics.

In [ ]:
validation_eras = sorted(validation["date"].unique())
scoring_eras = validation_eras[12:62]
assert len(scoring_eras) == 50

validation = (
    validation[validation["date"].isin(scoring_eras)]
    .dropna(subset=[TARGET])
    .set_index("numerai_ticker")
)
assert validation["date"].nunique() == 50

In [ ]:
!pip install -q --no-deps numerai-tools

In [ ]:
from tqdm import tqdm
from numerai_tools.scoring import neutral_correlation, neutral_contribution

scores = {}

for date, group in tqdm(
    validation.groupby("date"),
    total=validation["date"].nunique(),
):
    predictions = pd.DataFrame(
        model.predict(group[feature_cols]),
        index=group.index,
        columns=["prediction"],
    )

    # Predicate filters avoid loading the entire 300-column neutralizer at once.
    neutralizers = (
        pd.read_parquet(
            f"{DATASET_DIR}/validation_neutralizer.parquet",
            filters=[("date", "=", date)],
        )
        .set_index("numerai_ticker")
        .filter(like="neutralizer_")
        .dropna(axis=0, how="all")
    )
    reference = (
        pd.read_parquet(
            f"{DATASET_DIR}/validation_example_preds.parquet",
            filters=[("date", "=", date)],
        )
        .set_index("numerai_ticker")["signal"]
    )
    targets = group[TARGET]

    scores[date] = {
        "neutral_correlation": neutral_correlation(
            predictions, neutralizers, targets
        )["prediction"],
        "neutral_contribution_vs_example": neutral_contribution(
            predictions, reference, neutralizers, targets
        )["prediction"],
    }

In [ ]:
scores = pd.DataFrame.from_dict(scores, orient="index").sort_index()
scores.cumsum().plot(title="Cumulative v3 validation scores")

NCORR measures correlation after neutralization; the contribution column is only a proxy because the official Meta Model is not public. These starter features are mostly useful for understanding exposures, not as a complete Signals strategy. Compare the mean and standard deviation of the per-date scores:

In [ ]:
scores.mean() / scores.std()

Bring your own data to build a more original signal. For now, let's continue with how to structure a live submission.

# Live Submission

To make a live submission, you only need to submit a ticker column with its signal.

We accept the following tickers for live submissions:

* cusip
* sedol
* bloomberg_ticker
* composite_figi
* numerai_ticker

In [ ]:
from numerai_tools.scoring import tie_kept_rank

napi.download_dataset(f"{DATASET_DIR}/live.parquet")
live = pd.read_parquet(f"{DATASET_DIR}/live.parquet")

live["signal"] = model.predict(live[feature_cols])
# Rank predictions to the required [0, 1] range.
live["signal"] = tie_kept_rank(live[["signal"]])["signal"]

submission = live[["numerai_ticker", "signal"]].copy()
submission

In [ ]:
# Save and download your predictions.
filename = "signals_example_preds.csv"
submission.to_csv(filename, index=False)

# Download file if running in Google Colab.
try:
    from google.colab import files
    files.download(filename)
except ImportError:
    pass

Now you can visit the [Submissions page](https://signals.numer.ai/submissions) to submit these predictions to your model. Scores on the 60-day Jupiter target take approximately 13 weeks to resolve.